# Phase 13 Part 1A-R2 — Checkpoint-Selection Provenance Correction

**Model:** Gemini 3.8 Flash Medium · **Date:** 2026-09-21
**Manifest:** `phase4_v3` · **Taxonomy:** `kub_size_clean_v3_v1`
**Part 1A:** COMPLETE · **Part 1A-R1:** EXECUTION PASS · **Part 1A-R2 ONLY is authorized**

---

> **AMENDMENT PHASE ONLY — NO TRAINING / NO INFERENCE.**
>
> **NO TRAINING · NO OPTIMIZER · NO MODEL LOADING · NO FORWARD PASS · NO LOCALIZER
> INFERENCE · NO PROPOSALS · NO PREDICTED-REGION SEGMENTATION · NO 20-CONFIG GRID ·
> NO PART 1B · NO PHASE 14 · NO TEST ACCESS.**

## Why R2 exists

R1 froze an **8-term** checkpoint-selection tuple. Subsequent provenance verification
established that the historical **executing** trainer used a **4-term** tuple
(`scripts/training/train_phase13b_localizer.py` line 164). The clean-v3 primary experiment is
intended to reproduce the executing historical localizer mechanism as closely as scientifically
possible, so **the primary clean-v3 checkpoint-selection tuple is corrected to the executing
historical 4-term tuple**.

R2 supersedes **only** the R1 checkpoint-selection rule. All other R1 decisions remain frozen.

## Companion-notebook rule

Neither of these is modified by this task:

- `phase13_clean_v3_localizer_protocol_freeze.ipynb` — `68f787e1…`
- `phase13_clean_v3_localizer_supervisor_freeze_amendment.ipynb` — `6a08d6bd…`
- `phase13_clean_v3_localizer_protocol_frozen.json` — `999144bb…` (preserved for provenance)


In [1]:
import sys, json
from pathlib import Path
import pandas as pd
pd.set_option('display.max_rows', None); pd.set_option('display.max_columns', None)
pd.set_option('display.width', None); pd.set_option('display.max_colwidth', 100)

PROJECT_ROOT = Path(r'D:\Softfors\jup')
CODE_DIR = PROJECT_ROOT / 'notebooks/phasewise/phase_13_predicted_roi_localization/code'
for p in (str(PROJECT_ROOT), str(CODE_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import phase13_localizer_freeze_amendment_r2 as r2mod
TABLES = PROJECT_ROOT / 'reports/tables'
AUDITS = PROJECT_ROOT / 'reports/audits'

# Read-only regeneration. No model is loaded, no forward pass, no optimizer, no inference.
hashes = r2mod.write_all(write_frozen=True)
print(f'wrote {len(hashes)} artifacts')
print('training 0 | optimizer steps 0 | forward passes 0 | localizer inference 0 | '
      'proposals 0 | predicted-region inference 0 | grid search: not run | TEST reads 0')


wrote 3 artifacts
training 0 | optimizer steps 0 | forward passes 0 | localizer inference 0 | proposals 0 | predicted-region inference 0 | grid search: not run | TEST reads 0


## 1. Provenance comparison — four competing accounts of the selection rule

The project's source-precedence ladder ranks **executing source (rank 1)** above
**frozen protocol document (rank 2)** above **prose**. Four accounts exist historically.


In [2]:
import importlib
r1mod = importlib.import_module('phase13_localizer_freeze_amendment')

executing = r1mod.HISTORICAL_EXECUTING_SELECTION
protocol_doc = r1mod.HISTORICAL_PROTOCOL_DOC_SELECTION
stage3 = r1mod.HISTORICAL_STAGE3_SELECTION

rows = [
    {'account': 'EXECUTING CODE', 'authority_rank': 1, 'source': executing['source'],
     'line': executing['source_line'], 'length': executing['tuple_length'],
     'tuple': executing['score_tuple'], 'r2_role': 'PRIMARY (adopted)'},
    {'account': 'PHASE13B.md', 'authority_rank': 2, 'source': protocol_doc['source'],
     'line': '-', 'length': protocol_doc['tuple_length'],
     'tuple': protocol_doc['score_tuple'], 'r2_role': 'documentation only'},
    {'account': 'Stage-3 correction', 'authority_rank': 2, 'source': stage3['source'],
     'line': '-', 'length': stage3['tuple_length'],
     'tuple': stage3['score_tuple'], 'r2_role': 'documentation only'},
    {'account': 'R1 supervisor extension', 'authority_rank': 'n/a (supervisor decision)',
     'source': 'phase13_clean_v3_localizer_protocol_frozen.json', 'line': '-',
     'length': 8, 'tuple': '8-term lexicographic extension',
     'r2_role': 'SUPERSEDED_FOR_PRIMARY_CHECKPOINT_SELECTION (preserved as provenance)'},
]
display(pd.DataFrame(rows))


,account,authority_rank,source,line,length,tuple,r2_role
0,EXECUTING CODE,1,scripts/training/train_phase13b_localizer.py,164,4,"(tiny_recall90, overall_recall90, -mean_roi_area_fraction, -val_loss)",PRIMARY (adopted)
1,PHASE13B.md,2,PHASE13B.md,-,8,"(recovered frozen 16 Tiny misses@90, recovered frozen 7 global-empty@90, overall Tiny recall90, ...",documentation only
2,Stage-3 correction,2,reports/audits/phase13b_stage3_validation_correction.md,-,3,"(VAL Tiny recall90, overall component recall90, lower VAL localization loss)",documentation only
3,R1 supervisor extension,n/a (supervisor decision),phase13_clean_v3_localizer_protocol_frozen.json,-,8,8-term lexicographic extension,SUPERSEDED_FOR_PRIMARY_CHECKPOINT_SELECTION (preserved as provenance)


## 2. Executing 4-term source evidence

Read directly from the frozen historical trainer (plain file read — the file is **not**
imported, executed, or modified).


In [3]:
trainer = PROJECT_ROOT / r2mod.EXECUTING_TRAINER
print('file :', trainer.relative_to(PROJECT_ROOT))
print('SHA  :', r2mod.sha256_file(trainer))
print('line :', r2mod.EXECUTING_TRAINER_LINE)
print()
lines = trainer.read_text(encoding='utf-8').splitlines()
lo, hi = r2mod.EXECUTING_TRAINER_LINE - 4, r2mod.EXECUTING_TRAINER_LINE + 6
for i in range(lo, hi + 1):
    marker = '>>' if i == r2mod.EXECUTING_TRAINER_LINE else '  '
    print(f'{marker} {i:4d} | {lines[i-1]}')


file : scripts\training\train_phase13b_localizer.py
SHA  : 5985b9eb2fe43018babbd88004bf7e5b99d2783dcfa78c7404591a02bdcb555a
line : 164

    160 |   val_loss,search,cache,heatmap_diag=evaluate(model,val_loader,criterion,device,resolution,component_table)
    161 |   feasible_configs=search[search.feasible]
    162 |   if len(feasible_configs)>0:
    163 |    best_cfg=feasible_configs.sort_values(['tiny_recall90','overall_recall90','mean_roi_area_fraction','mean_proposals_per_image','threshold','top_k','margin'],ascending=[False,False,True,True,False,True,True]).iloc[0]
>>  164 |    score=(float(best_cfg.tiny_recall90),float(best_cfg.overall_recall90),-float(best_cfg.mean_roi_area_fraction),-float(val_loss))
    165 |    improved=best_score is None or score>best_score
    166 |    bad_epochs=0 if improved else bad_epochs+1
    167 |    if improved: best_score=score
    168 |   else:
    169 |    best_cfg=search.sort_values(['mean_roi_area_fraction','mean_proposals_per_image'],ascending=[

In [4]:
print('EXECUTING TUPLE (verbatim from the marked line):')
print('   (tiny_recall90, overall_recall90, -mean_roi_area_fraction, -val_loss)')
print('   length = 4')
print()
print('Same block confirms:')
print('  - improvement rule : score > best_score   (strict lexicographic)')
print('  - ties             : bad epoch')
print('  - patience         : 15')
print('  - best.pt          : saved only on improvement')
print('  - checkpoint key   : best_validation_score')
print('  - manifest recorded: phase4_v2')
print('  - proposal config  : best-of-the-20-config-grid PER EPOCH (the VAL-dependent')
print('                       behaviour that R1 correctly removed)')


EXECUTING TUPLE (verbatim from the marked line):
   (tiny_recall90, overall_recall90, -mean_roi_area_fraction, -val_loss)
   length = 4

Same block confirms:
  - improvement rule : score > best_score   (strict lexicographic)
  - ties             : bad epoch
  - patience         : 15
  - best.pt          : saved only on improvement
  - checkpoint key   : best_validation_score
  - manifest recorded: phase4_v2
  - proposal config  : best-of-the-20-config-grid PER EPOCH (the VAL-dependent
                       behaviour that R1 correctly removed)


## 3. R1 8-term vs R2 4-term

**Only** the checkpoint-selection rule changes. The hard-set / diagnostic fields move from
*selection terms* to *diagnostics* — a direct consequence of that single change.


In [5]:
R1_TUPLE = ['hard_tiny_recovered90', 'global_empty_images_covered90', 'tiny_recall90',
            'overall_recall90', 'multi_component_recall90', '-mean_roi_area_fraction',
            '-mean_proposals_per_image', '-val_loss']
R2_TUPLE = r2mod.PRIMARY_CHECKPOINT_TUPLE

print('R1 (8-term, SUPERSEDED for primary selection):')
for i, t in enumerate(R1_TUPLE, 1):
    tag = '' if t in R2_TUPLE else '   <-- no longer a selection term'
    print(f'  {i}. {t}{tag}')
print()
print('R2 (4-term, PRIMARY — executing historical):')
for i, t in enumerate(R2_TUPLE, 1):
    print(f'  {i}. {t}')
print()
print('label:', r2mod.PRIMARY_CHECKPOINT_TUPLE_LABEL)
print('provenance:', r2mod.PRIMARY_CHECKPOINT_SELECTION['provenance'])
print()
print('Removed from selection, retained as diagnostics:')
for k, v in r2mod.DIAGNOSTIC_METRICS.items():
    print(f'  {k}: {v}')


R1 (8-term, SUPERSEDED for primary selection):
  1. hard_tiny_recovered90   <-- no longer a selection term
  2. global_empty_images_covered90   <-- no longer a selection term
  3. tiny_recall90
  4. overall_recall90
  5. multi_component_recall90   <-- no longer a selection term
  6. -mean_roi_area_fraction
  7. -mean_proposals_per_image   <-- no longer a selection term
  8. -val_loss

R2 (4-term, PRIMARY — executing historical):
  1. tiny_recall90
  2. overall_recall90
  3. -mean_roi_area_fraction
  4. -val_loss

label: EXECUTING_HISTORICAL_4TERM_FIXED_CONFIG_SELECTION
provenance: HISTORICAL EXECUTING CODE

Removed from selection, retained as diagnostics:
  hard_tiny_recovered90: SECONDARY_DIAGNOSTIC_NOT_SELECTION_TERM
  global_empty_images_covered90: SECONDARY_DIAGNOSTIC_NOT_SELECTION_TERM
  multi_component_recall90: SECONDARY_DIAGNOSTIC_NOT_SELECTION_TERM
  mean_proposals_per_image: SECONDARY_DIAGNOSTIC_NOT_SELECTION_TERM


## 4. Recorded rationale (supervisor §6)


In [6]:
for k, v in r2mod.R2_RATIONALE.items():
    print(f'{k}. {v}')


A. Checkpoint selection is a material scientific parameter.
B. The executing historical trainer is higher-authority evidence than prose documentation.
C. Using the executing 4-term tuple minimizes unnecessary v2->v3 protocol drift.
D. The R1 8-term extension would make primary checkpoint selection depend partly on hard cohorts derived using the UNet++ historical-replication anchor.
E. UNet++ is NOT a Phase-13 reference model.
F. Keeping those fields as diagnostics preserves their analytical value without allowing them to bias primary checkpoint selection.


## 5. R1 → R2 additive diff

A new additive table — history is **not** rewritten. The R1 table
(`phase13_v2_to_clean_v3_localizer_protocol_diff.csv`) is untouched.


In [7]:
diff = pd.read_csv(r2mod.R2_DIFF)
counts = r2mod.diff_counts(diff)
display(diff[['field', 'change', 'scientifically_material', 'authorized',
              'material_change_group']])
print('counts:', counts)
assert counts['unauthorized_material_differences'] == 0
assert counts['r1_to_r2_material_differences'] == 1
assert counts['authorized_material_differences'] == 1
print()
print('Exactly ONE material difference (checkpoint_selection_tuple), authorized, 0 unauthorized.')


,field,change,scientifically_material,authorized,material_change_group
0,checkpoint_selection_tuple,CHANGED,True,True,M1
1,hard_set_selection_role,CHANGED,False,True,M1
2,proposal_configuration,SAME,False,False,NaN
3,primary_grid_search,SAME,False,False,NaN
4,historical_20_grid,SAME,False,False,NaN
5,heatmap_resolution_contract,SAME,False,False,NaN
6,architecture,SAME,False,False,NaN
7,target_generation,SAME,False,False,NaN
8,loss,SAME,False,False,NaN
9,optimizer,SAME,False,False,NaN


counts: {'r1_to_r2_material_differences': 1, 'authorized_material_differences': 1, 'unauthorized_material_differences': 0, 'changed_fields': 2, 'unchanged_fields': 17}

Exactly ONE material difference (checkpoint_selection_tuple), authorized, 0 unauthorized.


In [8]:
changed = diff[diff['change'] == 'CHANGED']
display(changed[['field', 'r1_value', 'r2_value', 'reason']])


,field,r1_value,r2_value,reason
0,checkpoint_selection_tuple,"8-term supervisor extension: (hard_tiny_recovered90, global_empty_images_covered90, tiny_recall9...","4-term executing historical tuple: (tiny_recall90, overall_recall90, -mean_roi_area_fraction, -v...",EXECUTING_CODE_PRECEDENCE_AND_MINIMIZE_PROTOCOL_DRIFT
1,hard_set_selection_role,checkpoint-selection terms,SECONDARY_DIAGNOSTIC_NOT_SELECTION_TERM (pre-frozen secondary diagnostic cohorts),CONSEQUENCE_OF_THE_SINGLE_AUTHORIZED_MATERIAL_CHANGE (checkpoint_selection_tuple)


## 6. Confirmation that all other protocol fields are unchanged

Programmatic key-level comparison of the R1 frozen JSON against the regenerated R2 protocol:
only an explicitly allow-listed set of keys may differ. Any other drift raises.


In [9]:
r2 = r2mod.build_r2_protocol()
unexpected = r2mod.verify_only_expected_keys_changed(r2)
print('unexpected protocol drift:', unexpected or 'NONE')
assert not unexpected, unexpected
print()
r1_frozen = json.loads(r2mod.R1_FROZEN_JSON.read_text(encoding='utf-8'))
same_keys = sorted(k for k in r1_frozen if r1_frozen.get(k) == r2.get(k))
print(f'identical top-level keys: {len(same_keys)}')
print('  ', ', '.join(same_keys))
print()
print('intentionally changed keys:')
print('  ', ', '.join(sorted(r2mod.EXPECTED_CHANGED_KEYS)))


unexpected protocol drift: NONE

identical top-level keys: 19
   coverage_contract, date, downstream_composition, downstream_geometry, downstream_model_scope, downstream_reference_model, epoch_provenance, gate_governance, heatmap_contract, label_defects, manifest_version, model, next_action, phase, phase14_distribution_requirement, population, proposal_configuration, retention_contract, taxonomy

intentionally changed keys:
   authorization, checkpoint_selection, checkpoint_selection_provenance, checkpoint_selection_r1, diagnostic_metrics, diff_from_r1, document_type, governance, hard_sets, next_action, part, pass_gate, r1_supersession, r2_rationale, sources, status, task


In [10]:
print('UNCHANGED scientific core (spot check against R1):')
for key in ['population', 'heatmap_contract', 'proposal_configuration', 'coverage_contract',
            'downstream_composition', 'downstream_geometry', 'downstream_model_scope',
            'downstream_reference_model', 'retention_contract', 'gate_governance',
            'phase14_distribution_requirement', 'label_defects', 'epoch_provenance']:
    same = r1_frozen.get(key) == r2.get(key)
    print(f'  {key:38s} identical={same}')
    assert same, key
print()
print('ALL LISTED CORE FIELDS IDENTICAL TO R1')


UNCHANGED scientific core (spot check against R1):
  population                             identical=True
  heatmap_contract                       identical=True
  proposal_configuration                 identical=True
  coverage_contract                      identical=True
  downstream_composition                 identical=True
  downstream_geometry                    identical=True
  downstream_model_scope                 identical=True
  downstream_reference_model             identical=True
  retention_contract                     identical=True
  gate_governance                        identical=True
  phase14_distribution_requirement       identical=True
  label_defects                          identical=True
  epoch_provenance                       identical=True

ALL LISTED CORE FIELDS IDENTICAL TO R1


## 7. Final R2 protocol display


In [11]:
cs = r2['checkpoint_selection']
print('STATUS:', r2['status'])
print('PART  :', r2['part'], '| DOC:', r2['document_type'])
print()
print('PRIMARY CHECKPOINT SELECTION:')
print('  label      :', cs['label'])
print('  provenance :', cs['provenance'], f"({cs['provenance_source']}:{cs['provenance_line']})")
print('  tuple      :', tuple(cs['tuple']))
print('  length     :', cs['tuple_length'])
print('  comparison :', cs['comparison'])
print('  ties       :', cs['ties'], '| patience:', cs['patience'])
print('  evaluated at:', cs['evaluated_at'])
print()
print('FIXED PROPOSAL CONFIG:', r2['proposal_configuration']['threshold'],
      r2['proposal_configuration']['margin'], r2['proposal_configuration']['top_k'],
      r2['proposal_configuration']['nms_iou'])
print('PRIMARY GRID SEARCH :', r2['proposal_configuration']['clean_v3_primary_grid_search'])
print('HISTORICAL 20-GRID  :', r2['proposal_configuration']['historical_20_grid'])
print()
print('HARD SETS: hard_tiny', r2['hard_sets']['hard_tiny']['count'],
      '| global_empty', r2['hard_sets']['global_empty']['count'],
      '| role', r2['hard_sets']['role'])
print('HEATMAP   :', r2['heatmap_contract']['input'], '->', r2['heatmap_contract']['output_heatmap'])
print('DOWNSTREAM:', r2['downstream_composition']['primary_name'],
      '| margin', r2['downstream_geometry']['config']['margin'])
print('RETENTION :', r2['downstream_geometry']['label'])
print('GATE      :', r2['gate_governance']['final_regional_gate'])


STATUS: SUPERVISOR_FROZEN_R2_READY_FOR_PART1B_TRAINING
PART  : 1A-R2 | DOC: CLEAN_V3_LOCALIZER_PROTOCOL_FROZEN_R2

PRIMARY CHECKPOINT SELECTION:
  label      : EXECUTING_HISTORICAL_4TERM_FIXED_CONFIG_SELECTION
  provenance : HISTORICAL EXECUTING CODE (scripts/training/train_phase13b_localizer.py:164)
  tuple      : ('tiny_recall90', 'overall_recall90', '-mean_roi_area_fraction', '-val_loss')
  length     : 4
  comparison : STRICT_LEXICOGRAPHIC_GREATER_THAN
  ties       : BAD_EPOCH | patience: 15
  evaluated at: ONE FIXED PROPOSAL CONFIGURATION ONLY (0.10 / 0.20 / 5 / 0.30)

FIXED PROPOSAL CONFIG: 0.1 0.2 5 0.3
PRIMARY GRID SEARCH : DISABLED
HISTORICAL 20-GRID  : DEFERRED_SECONDARY_SENSITIVITY_ONLY

HARD SETS: hard_tiny 15 | global_empty 5 | role PRE_FROZEN_SECONDARY_DIAGNOSTIC_COHORT
HEATMAP   : 512x512 -> 128x128
DOWNSTREAM: PREDICTED_REGION_FLOAT_GLOBAL_RETAINED | margin 0.2
RETENTION : PRACTICAL_GT_FREE_ORACLE_BENEFIT_RETENTION
GATE      : NOT_DECIDED


In [12]:
print('R1 SUPERSESSION:')
for k, v in r2['r1_supersession'].items():
    print(f'  {k}: {v}')


R1 SUPERSESSION:
  r1_artifact: reports/audits/phase13_clean_v3_localizer_protocol_frozen.json
  r1_artifact_sha256: 999144bbad4af4e1d2dc006279aef1a5fb93f450006ae2c5db159c791c913703
  r1_artifact_sha256_expected: 999144bbad4af4e1d2dc006279aef1a5fb93f450006ae2c5db159c791c913703
  r1_artifact_unchanged: True
  r1_md_artifact: reports/audits/phase13_clean_v3_localizer_protocol_frozen.md
  r1_status_for_execution: SUPERSEDED_BY_SUPERVISOR_R2_CHECKPOINT_SELECTION_CORRECTION
  r1_artifact_modified: False
  scope_of_supersession: ONLY the R1 8-term checkpoint-selection rule; all other R1 decisions remain frozen
  future_part1b_must_use: reports/audits/phase13_clean_v3_localizer_protocol_frozen_r2.json


## 8. Immutability verification (§17)


In [13]:
checks = [
    ('Part-1A notebook', r2mod.PART1A_NOTEBOOK, r2mod.PART1A_NOTEBOOK_SHA),
    ('R1 amendment notebook', r2mod.R1_NOTEBOOK, r2mod.R1_NOTEBOOK_SHA),
    ('R1 frozen JSON', r2mod.R1_FROZEN_JSON, r2mod.R1_FROZEN_JSON_SHA),
    ('hard-Tiny table', r2mod.TABLES / 'phase13_clean_v3_hard_tiny_replication_set.csv',
     '2f9828d77c008e7c90cd69e82d0af3568902ca9d508fdfcb9d8ccf8abd569538'),
    ('global-empty table', r2mod.TABLES / 'phase13_clean_v3_global_empty_replication_set.csv',
     'a0b332cad6bca4466ccc528cbe18cc527ef6503603ab69c55b9d699d7979f0ea'),
    ('historical history.csv', PROJECT_ROOT / 'experiments/phase13b_kub_localizer_r512_h128_'
     'sigmaeq075_centernetfocal_seed42_v1/history.csv',
     'dcdb4369d0c3a087fd0bd2611db0aac3ff0afd48db8c1f9a9bb5e79b834fdb76'),
    ('historical best.pt', PROJECT_ROOT / 'checkpoints/phase13b_kub_localizer_r512_h128_'
     'sigmaeq075_centernetfocal_seed42_v1/best.pt',
     'ece906df6c73f2e93c3f873661490bf3cb1a6b82be2d2d93b58c25bc967cbace'),
    ('historical latest.pt', PROJECT_ROOT / 'checkpoints/phase13b_kub_localizer_r512_h128_'
     'sigmaeq075_centernetfocal_seed42_v1/latest.pt',
     'a1461df2e8d6784984337c79b87c65833c39fed16382579088d97d7c60ba78f4'),
    ('historical Stage-4D notebook', PROJECT_ROOT / 'notebooks/phasewise/'
     'phase_13_predicted_roi_localization/phase13_stage4d_localizer_final_visual_analysis.ipynb',
     '561fd5d63d7fa0ea7de9155b055ce08035928d065054bb94b8a707ce83b969d0'),
]
rows = []
for name, path, expected in checks:
    actual = r2mod.sha256_file(path)
    rows.append({'artifact': name, 'expected': expected[:16] + '...',
                 'actual': actual[:16] + '...', 'unchanged': actual == expected})
    assert actual == expected, name
display(pd.DataFrame(rows))
print('ALL IMMUTABILITY CHECKS PASS')


,artifact,expected,actual,unchanged
0,Part-1A notebook,68f787e1f70fcf70...,68f787e1f70fcf70...,True
1,R1 amendment notebook,6a08d6bd43e173f4...,6a08d6bd43e173f4...,True
2,R1 frozen JSON,999144bbad4af4e1...,999144bbad4af4e1...,True
3,hard-Tiny table,2f9828d77c008e7c...,2f9828d77c008e7c...,True
4,global-empty table,a0b332cad6bca446...,a0b332cad6bca446...,True
5,historical history.csv,dcdb4369d0c3a087...,dcdb4369d0c3a087...,True
6,historical best.pt,ece906df6c73f2e9...,ece906df6c73f2e9...,True
7,historical latest.pt,a1461df2e8d67849...,a1461df2e8d67849...,True
8,historical Stage-4D notebook,561fd5d63d7fa0ea...,561fd5d63d7fa0ea...,True


ALL IMMUTABILITY CHECKS PASS


## 9. Zero-training / zero-inference boundary


In [14]:
for k, v in r2['governance'].items():
    print(f'  {k}: {v}')
g = r2['governance']
assert g['training'] == 0 and g['optimizer_steps'] == 0 and g['model_forward_passes'] == 0
assert g['localizer_inference'] == 0 and g['predicted_region_inference'] == 0
assert g['proposals_generated'] == 0 and g['test_content_reads'] == 0
assert not g['phase14_started'] and not g['historical_artifacts_modified']
assert not g['r1_artifacts_modified'] and not g['clean_v3_grid_search_run']
print()
print('BOUNDARY VERIFIED: no model was loaded, no forward pass, no optimizer,')
print('no inference, no proposals, no grid search, 0 TEST reads, 0 historical/R1 edits.')


  training: 0
  optimizer_steps: 0
  model_forward_passes: 0
  localizer_inference: 0
  predicted_region_inference: 0
  proposals_generated: 0
  clean_v3_grid_search_run: False
  test_content_reads: 0
  test_metrics: NONE
  phase14_started: False
  external_data_used: False
  experiment_a_modified: False
  winner_declared: False
  reference_model_selected: False
  regional_mechanism_promoted: False
  historical_artifacts_modified: False
  r1_artifacts_modified: False

BOUNDARY VERIFIED: no model was loaded, no forward pass, no optimizer,
no inference, no proposals, no grid search, 0 TEST reads, 0 historical/R1 edits.


## 10. Pass gate and artifact registry


In [15]:
pg = r2['pass_gate']
for k, v in pg.items():
    print(f'  {k}: {v}')
assert pg['unauthorized_material_differences'] == 0
assert all(v for k, v in pg.items() if k != 'unauthorized_material_differences')
print()
print('ALL PASS-GATE ITEMS TRUE (unauthorized material differences == 0)')


  r1_provenance_preserved: True
  r2_uses_executing_historical_4term_tuple: True
  fixed_proposal_config_retained: True
  primary_grid_search_remains_disabled: True
  hard_sets_retained_but_removed_from_checkpoint_selection: True
  hard_set_diagnostic_role_clearly_labelled: True
  unetpp_remains_replication_anchor_only: True
  heatmap_512_to_128_contract_unchanged: True
  scientific_core_otherwise_unchanged: True
  downstream_predicted_region_contract_unchanged: True
  no_training: True
  no_inference: True
  test_reads_zero: True
  part1b_not_authorized: True
  unauthorized_material_differences: 0

ALL PASS-GATE ITEMS TRUE (unauthorized material differences == 0)


In [16]:
reg = pd.DataFrame([{'artifact': k, 'sha256': v} for k, v in sorted(hashes.items())])
display(reg)
print()
print('FUTURE PART 1B MUST USE:', r2['r1_supersession']['future_part1b_must_use'])
print('NEXT ACTION:', r2['next_action'])


,artifact,sha256
0,phase13_clean_v3_localizer_protocol_frozen_r2.json,ab451e03366c589fae5c303ebf1382e086f4f115b97768eb36d491256a3b5e53
1,phase13_clean_v3_localizer_protocol_frozen_r2.md,b9bad4e583a6a46d4482ed5aa7bf30fd93c2299858183e46c61e3c7c4b18ff89
2,phase13_r1_to_r2_protocol_diff.csv,637bdffa4b6a51c2e5f1d5c00e5cfd59863e8a656f845a273ac2f3f67f8ca52c



FUTURE PART 1B MUST USE: reports/audits/phase13_clean_v3_localizer_protocol_frozen_r2.json
NEXT ACTION: WAIT_FOR_SUPERVISOR_PHASE13_PART1B_TRAINING_AUTHORIZATION


## 11. STOP

- Part 1A / 1A-R1 / 1A-R2: **COMPLETE**. Current executable protocol:
  **`SUPERVISOR_FROZEN_R2_READY_FOR_PART1B_TRAINING`**.
- Primary checkpoint selection: **4-term executing historical tuple** at the fixed proposal
  configuration. R1 8-term tuple: **SUPERSEDED_FOR_PRIMARY_CHECKPOINT_SELECTION** (provenance
  preserved).
- Hard-set / global-empty metrics: **`SECONDARY_DIAGNOSTIC_NOT_SELECTION_TERM`**.
- Part 1B: **ELIGIBLE / NOT AUTHORIZED**.
- `REGIONAL_INTEGRATION_GATE_AFTER_PHASE13`: **NOT_DECIDED** (no GO / REVISE_LOCALIZER / NO_GO).
- Phase 14: **not started**. `RA-MedFiLM-Seg++`: **not implemented**. TEST: **untouched**.

**STOP — WAIT_FOR_SUPERVISOR_PHASE13_PART1B_TRAINING_AUTHORIZATION.**


In [17]:
print('PHASE 13 PART 1A-R2 — COMPLETE (checkpoint-selection provenance correction)')
print('protocol status: SUPERVISOR_FROZEN_R2_READY_FOR_PART1B_TRAINING')
print('primary tuple: 4-term executing historical, at fixed config 0.10/0.20/5/0.30')
print('part1b: ELIGIBLE / NOT AUTHORIZED | gate: NOT_DECIDED | phase14: not started')
print('training 0 | inference 0 | proposals 0 | TEST reads 0')


PHASE 13 PART 1A-R2 — COMPLETE (checkpoint-selection provenance correction)
protocol status: SUPERVISOR_FROZEN_R2_READY_FOR_PART1B_TRAINING
primary tuple: 4-term executing historical, at fixed config 0.10/0.20/5/0.30
part1b: ELIGIBLE / NOT AUTHORIZED | gate: NOT_DECIDED | phase14: not started
training 0 | inference 0 | proposals 0 | TEST reads 0
